In [1]:
!pip install datasets transformers --quiet

In [2]:
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# Load the dataset
ds = load_dataset("edaschau/bitcoin_news")
df = ds['train'].to_pandas()

# Data filtering
df['date'] = pd.to_datetime(df['time_unix'], unit="s")
df = df.sort_values(by='date', ascending=True)
df.set_index('date', inplace=True)

df = df[['title', 'article_text', 'url']]

# print(df.head(3))
print(df.tail(3))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


                                                                 title  \
date                                                                     
2024-01-24 14:36:34  ‘Alameda Gap’ Keeps Bitcoin Volatile Even as E...   
2024-01-24 20:11:00  Are cryptocurrency rewards credit cards a good...   
2024-01-24 21:47:18  Nasdaq 100 Gains for Fifth Day; Treasuries Dec...   

                                                          article_text  \
date                                                                     
2024-01-24 14:36:34  (Bloomberg) -- A dearth of liquidity in the di...   
2024-01-24 20:11:00  Key takeaways Crypto rewards cards offer a new...   
2024-01-24 21:47:18  (Bloomberg) -- US stock indexes ended Wednesda...   

                                                                   url  
date                                                                    
2024-01-24 14:36:34  https://finance.yahoo.com/news/alameda-gap-kee...  
2024-01-24 20:11:00  https://finance.ya

In [3]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")
model = AutoModelForSequenceClassification.from_pretrained("mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis")

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/933 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

In [4]:
def analyze_sentiment_in_batches(texts, batch_size):
    sentiment_labels = []

    for i in tqdm(range(0, len(texts), batch_size)):  # Live progress bar
        batch_texts = texts[i:i + batch_size]

        # Tokenize and move inputs to GPU
        inputs = tokenizer(batch_texts, return_tensors="pt", truncation=True, max_length=512, padding=True)
        inputs = {key: val.to(device) for key, val in inputs.items()}  # Move tensors to GPU

        # Perform inference on GPU
        with torch.no_grad():
            outputs = model(**inputs)

        # Get predicted class labels
        predicted_classes = torch.argmax(outputs.logits, dim=1)
        batch_labels = [model.config.id2label[idx.item()] for idx in predicted_classes]
        sentiment_labels.extend(batch_labels)

    return sentiment_labels

# Ensure model and device are set correctly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

# Run with live progress bar
df['sentiment_labels'] = analyze_sentiment_in_batches(df['article_text'].tolist(), batch_size=256)

Using device: cuda


100%|██████████| 316/316 [24:10<00:00,  4.59s/it]


In [5]:
df.head(3)

,title,article_text,url,sentiment_labels
date,,,,
2011-06-22 10:56:00,Compromised account leads to massive Bitcoin s...,"Bitcoin, for those not aware, is a completely ...",https://finance.yahoo.com/news/2011-06-22-comp...,negative
2012-02-01 18:02:32,Bitcoin May Be The Currency Of The Future,Have you heard of Bitcoin? If you're a fan of ...,https://finance.yahoo.com/news/bitcoin-may-cur...,neutral
2012-03-22 19:23:56,Should Africa Adopt a Shared Currency? And Sho...,Dekstop /Flickr I wrote on Monday about Sweden...,https://finance.yahoo.com/news/africa-adopt-sh...,positive


In [7]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Save the dataset to a CSV file in Google Drive
df.to_csv('/content/drive/My Drive/Colab Notebooks/bitcoin_news_data.csv')

Mounted at /content/drive
